# B05 · Sesión 6 — El sistema experto como controlador

**Objetivo (RA5-d/e):** implementar un **controlador experto** de climatización, medir sus **especificaciones de respuesta** y compararlo con un PID.

> Práctica guiada de la Sesión 6 de los [apuntes](../apuntes.md).

In [ ]:
%pip install experta

# Parche obligatorio en Python 3.10+ (collections.Mapping se eliminó)
import collections, collections.abc
if not hasattr(collections, "Mapping"):
    collections.Mapping = collections.abc.Mapping
    collections.Iterable = collections.abc.Iterable
    collections.MutableMapping = collections.abc.MutableMapping

from experta import *

## 1. La planta

Modelo simple con inercia térmica. Equilibrio: `temp = exterior + 10·potencia`.

In [ ]:
def planta(temp, potencia, inercia=0.05, exterior=15.0):
    return temp + inercia * (potencia * 10 - (temp - exterior))

## 2. El controlador experto

!!! warning "`self` NO puede usarse dentro de `P(...)`"
    El decorador se evalúa al definir la clase. Declara el **error** como hecho en `DefFacts` (donde `self` sí existe) y compara con constantes.

In [ ]:
class ControladorClima(KnowledgeEngine):
    def __init__(self, setpoint=21.0):
        super().__init__()
        self.setpoint, self.potencia, self._temp = setpoint, 0.0, 15.0

    @DefFacts()
    def _hechos(self):
        yield Fact(error=self.setpoint - self._temp)

    @Rule(Fact(error=P(lambda e: e > 3)))
    def alta(self): self.potencia = 1.0

    @Rule(Fact(error=P(lambda e: 1 < e <= 3)))
    def media(self): self.potencia = 0.8

    @Rule(Fact(error=P(lambda e: 0 < e <= 1)))
    def baja(self): self.potencia = 0.6

    @Rule(Fact(error=P(lambda e: e <= 0)))
    def apagar(self): self.potencia = 0.0

    def paso(self, temp):
        self._temp = temp
        self.reset(); self.run()
        return self.potencia

ctrl = ControladorClima(21.0)
temp, hist = 15.0, []
for i in range(60):
    potencia = ctrl.paso(temp)
    temp = planta(temp, potencia)
    hist.append(temp)
    if i % 10 == 0:
        print(f"t={i:2d}  temp={temp:5.2f}  potencia={potencia}")

## 3. Mide las especificaciones de respuesta

- **Error en régimen permanente:** temperatura final − setpoint.
- **Tiempo de asentamiento:** primer paso en que entra y se queda en la banda 20,5-21,5 ºC.
- **Sobreimpulso:** máximo por encima de 21,5 ºC.

In [ ]:
# TODO: calcula las tres especificaciones sobre `hist`
banda = [i for i, t in enumerate(hist) if 20.5 <= t <= 21.5]
print("error regimen permanente:", round(hist[-1] - 21.0, 2))
print("asentamiento (paso):", banda[0] if banda else None)
print("sobreimpulso:", round(max(hist) - 21.0, 2))

## 4. Perturbación

En el paso 30, baja la temperatura exterior a 5 ºC (alguien abre una ventana) y observa si se recupera.

In [ ]:
# TODO: repite la simulacion con exterior=5.0 a partir del paso 30
...

## 5. Sensibilidad y comparación con PID

Cambia el umbral de la regla `media` (de 3 a 1,5) y repite la tabla. ¿Qué pasa con el sobreimpulso y el asentamiento?

| Especificación | Umbral 3 | Umbral 1,5 |
|---|---|---|
| Error en régimen permanente | | |
| Tiempo de asentamiento | | |
| Sobreimpulso | | |

**Para casa:** describe cuándo usarías un PID y cuándo un controlador experto o difuso. Justifica con los números que has obtenido.